# **Classificazione di immagini aeree**
Scopo di questa esercitazione è l’implementazione di una tecnica di **classificazione di immagini aeree** sulla base di feature di tessitura estratte da **matrici di co-occorrenza** dei livelli di grigio.

In particolare, data in input un’**immagine**, l’algoritmo dovrà restituire in output la classe di appartenenza tra le quattro esistenti nel training set (Fields, Houses, River, Wood).

<img src=https://biolab.csr.unibo.it/vr/esercitazioni/NotebookImages/EsClassificazioneImmaginiAeree/Classificazione.png width="800">

Il sistema di classificazione necessita di una **fase preliminare di training** durante la quale viene addestrato il classificatore a partire da un insieme di immagini di esempio delle diverse classi.


# **Import delle librerie**
È necessario ora eseguire l'import delle librerie utilizzate durante l'esercitazione.

In [ ]:
import cv2
import numpy as np
import math
import glob
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from google.colab.patches import cv2_imshow
from sklearn.metrics import confusion_matrix
from tqdm.notebook import tqdm

# **Utility functions**

Eseguire il codice seguente per definire alcune funzioni di utilità utilizzate nell'esercitazione:
*   `load_data` carica in una lista i **nomi delle immagini** presenti in una data cartella e determina, a partire dal primo carattere del nome dell'immagine, la **classe** di appartenenza.
*   `show_images` visualizza un insieme di immagini riportando per ciascuna la classe (ground truth o assegnata dal classificatore).
*   `show_confusion_matrix` visualizza la matrice di confusione.

In [ ]:
def load_data(folder):
  img_list = [file for file in glob.glob(os.path.join(folder,'*.jpg'))]
  labels = np.array([(int)(os.path.splitext(os.path.basename(img))[0][0])-2 for img in img_list])
  return img_list, labels

def show_images(images, labels, correct):
  rows = 5
  columns = 8
  label_names = ['Fields', 'Houses', 'River', 'Wood']
  plt.rcParams.update({'font.size': 20})
  _, axs = plt.subplots(rows, columns, squeeze=False, figsize=(30, 15))

  for i in range(rows*columns):
    r = i // columns
    c = i % columns
    axs[r, c].set_title(label_names[labels[i]])
    axs[r, c].axis('off')
    if (correct[i]):
      axs[r, c].imshow(cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB))
    else:
       rect = patches.Rectangle(
       (0, 0),                    # position (x, y)
       images[i].shape[1]-1,      # width
       images[i].shape[0]-1,      # height
       linewidth=5,               # thickness
       edgecolor='red',
       facecolor='none'
       )
       axs[r, c].add_patch(rect)
       axs[r, c].imshow(cv2.cvtColor(images[i], cv2.COLOR_BGR2GRAY), cmap='gray', vmin=0, vmax=255)

def show_confusion_matrix(conf_matrix,class_names,figsize=(10,10)):
  fig, ax = plt.subplots(figsize=figsize)
  img=ax.matshow(conf_matrix)
  tick_marks = np.arange(len(class_names))
  _=plt.xticks(tick_marks, class_names,rotation=45)
  _=plt.yticks(tick_marks, class_names)
  _=plt.ylabel('Real')
  _=plt.xlabel('Predicted')

  for i in range(len(class_names)):
    for j in range(len(class_names)):
        text = ax.text(j, i, '{0:.1%}'.format(conf_matrix[i, j]),
                       ha='center', va='center', color='w')

# **Dataset**
Prima di incominciare, è necessario scaricare il dataset di immagini aeree.

Eseguendo la cella sottostante tutto il materiale necessario per lo svolgimento dell'esercitazione verrà scaricato sulla macchina remota. Alla fine dell'esecuzione selezionare nel menù laterale **File** per verificare che tutto sia stato scaricato correttamente.

In [ ]:
!wget http://bias.csr.unibo.it/VR/Esercitazioni/DBs/AerialImages.zip
!unzip /content/AerialImages.zip
!rm /content/AerialImages.zip

In [ ]:
img_list, labels = load_data('/content/AerialImages/Training')
images_color = [cv2.imread(img) for img in img_list]
show_images(images_color, labels, np.full(labels.shape, True))

# **Training del classificatore**
I passaggi preliminari per la preparazione del training set sono i seguenti:
*   Conversione delle immagini di training in immagini **grayscale**
*   Calcolo, per ciascuna immagine, della corrispondente **matrice di co-occorrenza non orientata**

Nella porzione di codice che segue vengono anche visualizzate le matrici di co-occorrenza non orientate calcolate per le immagini di training.


### ***Calcolo della matrice di co-occorrenza per una specifica direzione***
La funzione `cooccurrence_matrix` calcola la **matrice di co-occorrenza nella direzione `d=(dx, dy)`** a partire da un'immagine grayscale in input `img`.

L'elemento $M[i,j]$ rappresenta la probabilità che, dato un generico pixel $[x,y]$ con valore di intensità luminosa $i$, il pixel in posizione $[x+dx, y+dy]$ abbia valore di intensità luminosa $j$.

### Suggerimenti

*   Inserire nel codice i controlli necessari per **non uscire dall'immagine** quando si applica lo spostamento $d$.
*   La matrice risultante dev'essere **indipendente dalla dimensione dell'immagine**. È dunque necessario normalizzare gli elementi della matrice rispetto al numero di pixel usati per il conteggio.


In [ ]:
def cooccurrence_matrix(img, d):
  # Computation of the co-occurrence matrix for direction d=(dy, dx)
  # TODO
  return co_occ_mat

### ***Calcolo della matrice di co-occorrenza non orientata***
La funzione `unoriented_cooccurrence_matrix` calcola la **matrice di co-occorrenza non orientata** per l'immagine in input `img`.
I passaggi per il calcolo della matrice sono i seguenti:

1.   Calcolo delle **4 matrici orientate** nelle direzioni `[(0, -1), (-1,-1), (-1, 0), (-1, 1)]`
2.   Calcolo della matrice non orientata come **media delle 4 matrici direzionali**.

In [ ]:
def unoriented_cooccurrence_matrix(img):
  #Computation of the unoriented co-occurrence matrix
  # TODO
  return u_co_occ_mat

In [ ]:
gray_images = [cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in images_color]
print('Unoriented co-occurence matrices computation')
matrices = [unoriented_cooccurrence_matrix(img) for img in tqdm(gray_images)]
matrices_images = [np.array(255*m/m.max(), dtype=np.uint8) for m in matrices]
show_images(matrices_images, labels, np.full(labels.shape, True))

### ***Estrazione delle feature e normalizzazione***
La funzione `extract_features` estrae un vettore di feature (array numpy con 4 elementi) a partire da una matrice di co-occorrenza non orientata `M`.

Il vettore di feature conterrà i valori di *energia*, *entropia*, *contrasto* e *omogeneità* calcolati come segue:

*   Energia $\sqrt{\sum_{i,j}M[i,j]^2}$
*   Entropia $-\sum_{i,j}M[i,j] \cdot log M[i,j]$
*   Contrasto $\sum_{i,j}(i-j)^2 \cdot M[i,j]$
*   Omogeneità $\sum_{i,j}\frac{M[i,j]}{1+\mid i-j \mid}$

### Suggerimenti
Attenzione all'uso della funzione logaritmo...




In [ ]:
def extract_features(M):
  energy = 0.0
  entropy = 0.0
  contrast = 0.0
  homogeneity = 0.0

  # TODO
  return np.array([energy, entropy, contrast, homogeneity])

Poiché i valori delle feature calcolate dalla funzione extract_features sono definiti in range differenti è bene procedere a una **normalizzazione** dei valori feature per feature (funzione `normalize`).

Per la normalizzazione è possibile sfruttare i **valori massimi** (`max_vals`) e **minimi** (`min_vals`) osservati in fase di **training** per le singole feature, e riscalare di conseguenza tutti i vettori.

In [ ]:
def normalize(features, min_vals, max_vals):
  ranges = max_vals - min_vals
  features = np.subtract(features, min_vals)
  features = np.divide(features, ranges)
  return features

### **Costruzione del training set**
Il training set viene costruito attraverso i passaggi seguenti:
*   **Estrazione delle feature** dalle matrici di co-occorrenza non orientate delle immagini di training
*   Determinazione dei **valori minimi e massimi** di ciascuna feature sul training set per la normalizzazione
*   **Normalizzazione** dei vettori di feature sulla base dei valori determinati al passo precedente



In [ ]:
print('Feature extraction')
features = [extract_features(m) for m in tqdm(matrices)]
min_vals = np.min(features, axis=0)
max_vals = np.max(features, axis=0)
features = normalize(features, min_vals, max_vals)

### **Training del classificatore**
Il modello usato per la classificazione è un **classificatore KNN** messo a disposizione dalla libreria opencv.

La documentazione della libreria opencv sul classificatore KNN è disponibile [qui](https://docs.opencv.org/4.x/dd/de1/classcv_1_1ml_1_1KNearest.html).

Si suggerisce di provare a variare il parametro $K$ del classificatore per valutarne l'impatto sulle prestazioni.

In [ ]:
k=3
knn=cv2.ml.KNearest.create()
knn.setDefaultK(k)
knn.train(features.astype(np.float32), cv2.ml.ROW_SAMPLE, labels.astype(np.float32))

# **Test del classificatore**
Il classificatore KNN addestrato precedentemente viene ora utilizzato per la classificazione delle immagini di test.

Il testing set viene predisposto seguendo gli stessi passaggi utilizzati per il training set.

In [ ]:
test_img_list, test_labels = load_data('/content/AerialImages/Test')
test_color_images = [cv2.imread(img) for img in test_img_list]
print('Unoriented co-occurence matrices computation')
test_matrices = [unoriented_cooccurrence_matrix(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)) for img in tqdm(test_color_images)]
print('Feature extraction')
test_features = [extract_features(m) for m in tqdm(test_matrices)]
test_features = normalize(test_features, min_vals, max_vals)
print('Prediction')
_,results=knn.predict(test_features.astype(np.float32))

Visualizzazione delle immagini di test, delle corrispondenti matrici di co-occorrenza non orientate e del **risultato della classificazione** (label predetta dal classificatore).

Le immagini classificate erroneamente sono visualizzate in grayscale.

In [ ]:
results_int = results[:,0].astype(int)
show_images(test_color_images, results_int, (results_int==test_labels))
matrices_images = [np.array(255*m/m.max(), dtype=np.uint8) for m in test_matrices]
show_images(matrices_images, results_int, np.full(test_labels.shape, True))

# **Valutazione delle prestazioni**
Per una visualizzazione più chiara del comportamento del classificatore si può calcolare la **matrice di confusione** che
offre una rappresentazione dell'accuratezza di classificazione statistica.

Ogni colonna della matrice rappresenta la classe predetta dal sistema, mentre ogni riga rappresenta la classe reale. L'elemento sulla riga i e sulla colonna j è il numero di casi in cui il classificatore ha classificato un'immagine di classe i come appartenente alla classe j. Attraverso questa matrice è osservabile se vi è "confusione" nella classificazione di diverse classi.

La funzione `plot_confusion_matrix` visualizza graficamente la matrice di confusione.

Per il calcolo della matrice di confusione si utilizza una funzionalità della libreria pandas.

La matrice di confusione viene calcolata confrontando il ground truth (label vere associate ai pattern di test) e il risultato della classificazione fornito dal classificatore KNN.

In [ ]:
correct = np.equal(results_int, test_labels)
accuracy=correct.sum()/len(correct)
print('Test set accuracy: {:.3f}'.format(accuracy))

In [ ]:
conf_matrix=confusion_matrix(test_labels, results_int, normalize='true')
show_confusion_matrix(conf_matrix, ['Fields', 'Houses', 'River', 'Wood'])